# HotelInitialLoad

Materialize the deterministic hotel CSV corpus into exactly two authoritative lakehouses,
enforce schema/type/key expectations, derive control tables, and fail loudly on contract
violations.

In [ ]:
correlation_id = "00000000-0000-0000-0000-000000000000"
source_root = "data/generated"
load_mode = "full"
reset_before_load = True
workspace_id = "00000000-0000-0000-0000-000000000000"
workspace_name = "HotelAgent.Workspace"
hotel_operations_lakehouse_id = "00000000-0000-0000-0000-000000000000"
hotel_guest_services_lakehouse_id = "00000000-0000-0000-0000-000000000000"

In [ ]:
from __future__ import annotations

import json
import hashlib
from datetime import date, datetime
from decimal import Decimal
from pathlib import Path

import notebookutils
from delta.tables import DeltaTable
from pyspark.errors import AnalysisException
from pyspark.sql import DataFrame, functions as F, types as T

DATE_START = date(2026, 3, 1)
DATE_END = date(2027, 3, 31)

SOURCE_TABLE_SPECS: dict[str, dict[str, object]] = {
    "properties": {
        "lakehouse": "operations",
        "primary_key": ["property_id"],
        "schema": T.StructType([
            T.StructField("property_id", T.StringType(), False),
            T.StructField("name", T.StringType(), False),
            T.StructField("region", T.StringType(), False),
            T.StructField("city", T.StringType(), False),
            T.StructField("country", T.StringType(), False),
            T.StructField("currency", T.StringType(), False),
            T.StructField("tax_rate", T.DecimalType(10, 4), False),
            T.StructField("facilities", T.StringType(), False),
            T.StructField("pool_hours", T.StringType(), False),
            T.StructField("parking", T.StringType(), False),
            T.StructField("transport", T.StringType(), False),
        ]),
    },
    "room_types": {
        "lakehouse": "operations",
        "primary_key": ["room_type_id"],
        "schema": T.StructType([
            T.StructField("room_type_id", T.StringType(), False),
            T.StructField("property_id", T.StringType(), False),
            T.StructField("name", T.StringType(), False),
            T.StructField("capacity", T.IntegerType(), False),
            T.StructField("accessible", T.BooleanType(), False),
            T.StructField("base_rate", T.DecimalType(10, 2), False),
        ]),
    },
    "rooms": {
        "lakehouse": "operations",
        "primary_key": ["room_id"],
        "schema": T.StructType([
            T.StructField("room_id", T.StringType(), False),
            T.StructField("property_id", T.StringType(), False),
            T.StructField("room_type_id", T.StringType(), False),
            T.StructField("room_number", T.StringType(), False),
            T.StructField("room_status", T.StringType(), False),
        ]),
    },
    "amenities": {
        "lakehouse": "operations",
        "primary_key": ["amenity_id"],
        "schema": T.StructType([
            T.StructField("amenity_id", T.StringType(), False),
            T.StructField("name", T.StringType(), False),
        ]),
    },
    "property_amenities": {
        "lakehouse": "operations",
        "primary_key": ["property_id", "amenity_id"],
        "schema": T.StructType([
            T.StructField("property_id", T.StringType(), False),
            T.StructField("amenity_id", T.StringType(), False),
        ]),
    },
    "daily_inventory": {
        "lakehouse": "operations",
        "primary_key": ["property_id", "room_type_id", "service_date"],
        "date_columns": ["service_date"],
        "schema": T.StructType([
            T.StructField("property_id", T.StringType(), False),
            T.StructField("room_type_id", T.StringType(), False),
            T.StructField("service_date", T.DateType(), False),
            T.StructField("physical_rooms", T.IntegerType(), False),
            T.StructField("out_of_order_rooms", T.IntegerType(), False),
            T.StructField("sellable_rooms", T.IntegerType(), False),
            T.StructField("occupied_rooms", T.IntegerType(), False),
        ]),
    },
    "daily_rates": {
        "lakehouse": "operations",
        "primary_key": ["property_id", "room_type_id", "service_date", "rate_plan"],
        "date_columns": ["service_date"],
        "schema": T.StructType([
            T.StructField("property_id", T.StringType(), False),
            T.StructField("room_type_id", T.StringType(), False),
            T.StructField("service_date", T.DateType(), False),
            T.StructField("rate_plan", T.StringType(), False),
            T.StructField("currency", T.StringType(), False),
            T.StructField("rate", T.DecimalType(10, 2), False),
        ]),
    },
    "property_daily_metrics": {
        "lakehouse": "operations",
        "primary_key": ["property_id", "service_date"],
        "date_columns": ["service_date"],
        "schema": T.StructType([
            T.StructField("property_id", T.StringType(), False),
            T.StructField("service_date", T.DateType(), False),
            T.StructField("sellable_rooms", T.IntegerType(), False),
            T.StructField("occupied_rooms", T.IntegerType(), False),
            T.StructField("room_revenue", T.DecimalType(14, 2), False),
            T.StructField("currency", T.StringType(), False),
        ]),
    },
    "outlets": {
        "lakehouse": "operations",
        "primary_key": ["outlet_id"],
        "schema": T.StructType([
            T.StructField("outlet_id", T.StringType(), False),
            T.StructField("property_id", T.StringType(), False),
            T.StructField("name", T.StringType(), False),
            T.StructField("outlet_type", T.StringType(), False),
            T.StructField("capacity", T.IntegerType(), False),
        ]),
    },
    "guests": {
        "lakehouse": "guest_services",
        "primary_key": ["guest_id"],
        "schema": T.StructType([
            T.StructField("guest_id", T.StringType(), False),
            T.StructField("first_name", T.StringType(), False),
            T.StructField("last_name", T.StringType(), False),
            T.StructField("email", T.StringType(), False),
            T.StructField("country", T.StringType(), False),
            T.StructField("privacy_consent", T.BooleanType(), False),
        ]),
    },
    "reservations": {
        "lakehouse": "guest_services",
        "primary_key": ["reservation_id"],
        "date_columns": ["check_in", "check_out"],
        "schema": T.StructType([
            T.StructField("reservation_id", T.StringType(), False),
            T.StructField("guest_id", T.StringType(), False),
            T.StructField("property_id", T.StringType(), False),
            T.StructField("room_type_id", T.StringType(), False),
            T.StructField("check_in", T.DateType(), False),
            T.StructField("check_out", T.DateType(), False),
            T.StructField("status", T.StringType(), False),
            T.StructField("cancellation_policy", T.StringType(), False),
            T.StructField("room_count", T.IntegerType(), False),
            T.StructField("rate_total", T.DecimalType(14, 2), False),
            T.StructField("currency", T.StringType(), False),
            T.StructField("channel", T.StringType(), False),
        ]),
    },
    "stays": {
        "lakehouse": "guest_services",
        "primary_key": ["stay_id"],
        "schema": T.StructType([
            T.StructField("stay_id", T.StringType(), False),
            T.StructField("reservation_id", T.StringType(), False),
            T.StructField("property_id", T.StringType(), False),
            T.StructField("guest_id", T.StringType(), False),
            T.StructField("room_id", T.StringType(), False),
            T.StructField("status", T.StringType(), False),
            T.StructField("actual_check_in", T.TimestampType(), False),
            T.StructField("actual_check_out", T.TimestampType(), True),
        ]),
    },
    "restaurant_reservations": {
        "lakehouse": "guest_services",
        "primary_key": ["restaurant_reservation_id"],
        "schema": T.StructType([
            T.StructField("restaurant_reservation_id", T.StringType(), False),
            T.StructField("guest_id", T.StringType(), False),
            T.StructField("property_id", T.StringType(), False),
            T.StructField("outlet_id", T.StringType(), False),
            T.StructField("reservation_at", T.TimestampType(), False),
            T.StructField("party_size", T.IntegerType(), False),
            T.StructField("status", T.StringType(), False),
        ]),
    },
    "folios": {
        "lakehouse": "guest_services",
        "primary_key": ["folio_id"],
        "schema": T.StructType([
            T.StructField("folio_id", T.StringType(), False),
            T.StructField("stay_id", T.StringType(), False),
            T.StructField("reservation_id", T.StringType(), False),
            T.StructField("guest_id", T.StringType(), False),
            T.StructField("property_id", T.StringType(), False),
            T.StructField("currency", T.StringType(), False),
            T.StructField("total_charges", T.DecimalType(14, 2), False),
            T.StructField("total_adjustments", T.DecimalType(14, 2), False),
            T.StructField("total_payments", T.DecimalType(14, 2), False),
            T.StructField("total_refunds", T.DecimalType(14, 2), False),
            T.StructField("balance", T.DecimalType(14, 2), False),
        ]),
    },
    "folio_transactions": {
        "lakehouse": "guest_services",
        "primary_key": ["folio_transaction_id"],
        "schema": T.StructType([
            T.StructField("folio_transaction_id", T.StringType(), False),
            T.StructField("folio_id", T.StringType(), False),
            T.StructField("transaction_type", T.StringType(), False),
            T.StructField("description", T.StringType(), False),
            T.StructField("amount", T.DecimalType(14, 2), False),
            T.StructField("currency", T.StringType(), False),
            T.StructField("posted_at", T.TimestampType(), False),
        ]),
    },
    "payments": {
        "lakehouse": "guest_services",
        "primary_key": ["payment_id"],
        "schema": T.StructType([
            T.StructField("payment_id", T.StringType(), False),
            T.StructField("folio_id", T.StringType(), False),
            T.StructField("payment_method", T.StringType(), False),
            T.StructField("payment_token", T.StringType(), False),
            T.StructField("card_mask", T.StringType(), False),
            T.StructField("amount", T.DecimalType(14, 2), False),
            T.StructField("currency", T.StringType(), False),
            T.StructField("payment_status", T.StringType(), False),
        ]),
    },
    "billing_disputes": {
        "lakehouse": "guest_services",
        "primary_key": ["billing_dispute_id"],
        "schema": T.StructType([
            T.StructField("billing_dispute_id", T.StringType(), False),
            T.StructField("folio_id", T.StringType(), False),
            T.StructField("dispute_type", T.StringType(), False),
            T.StructField("status", T.StringType(), False),
            T.StructField("amount", T.DecimalType(14, 2), False),
            T.StructField("opened_at", T.TimestampType(), False),
        ]),
    },
    "special_requests": {
        "lakehouse": "guest_services",
        "primary_key": ["special_request_id"],
        "schema": T.StructType([
            T.StructField("special_request_id", T.StringType(), False),
            T.StructField("reservation_id", T.StringType(), False),
            T.StructField("guest_id", T.StringType(), False),
            T.StructField("request_type", T.StringType(), False),
            T.StructField("status", T.StringType(), False),
            T.StructField("requested_at", T.TimestampType(), False),
            T.StructField("details", T.StringType(), False),
        ]),
    },
    "maintenance_requests": {
        "lakehouse": "operations",
        "primary_key": ["maintenance_request_id"],
        "schema": T.StructType([
            T.StructField("maintenance_request_id", T.StringType(), False),
            T.StructField("property_id", T.StringType(), False),
            T.StructField("stay_id", T.StringType(), False),
            T.StructField("room_id", T.StringType(), False),
            T.StructField("request_type", T.StringType(), False),
            T.StructField("status", T.StringType(), False),
            T.StructField("reported_at", T.TimestampType(), False),
        ]),
    },
    "work_orders": {
        "lakehouse": "operations",
        "primary_key": ["work_order_id"],
        "schema": T.StructType([
            T.StructField("work_order_id", T.StringType(), False),
            T.StructField("maintenance_request_id", T.StringType(), False),
            T.StructField("property_id", T.StringType(), False),
            T.StructField("status", T.StringType(), False),
            T.StructField("priority", T.StringType(), False),
            T.StructField("opened_at", T.TimestampType(), False),
        ]),
    },
    "action_confirmations": {
        "lakehouse": "guest_services",
        "primary_key": ["confirmation_id"],
        "schema": T.StructType([
            T.StructField("confirmation_id", T.StringType(), False),
            T.StructField("action_type", T.StringType(), False),
            T.StructField("target_id", T.StringType(), False),
            T.StructField("status", T.StringType(), False),
            T.StructField("confirmed_at", T.TimestampType(), False),
        ]),
    },
    "idempotency_records": {
        "lakehouse": "guest_services",
        "primary_key": ["idempotency_key"],
        "schema": T.StructType([
            T.StructField("idempotency_key", T.StringType(), False),
            T.StructField("action_type", T.StringType(), False),
            T.StructField("result_id", T.StringType(), False),
            T.StructField("created_at", T.TimestampType(), False),
        ]),
    },
    "audit_events": {
        "lakehouse": "guest_services",
        "primary_key": ["audit_event_id"],
        "schema": T.StructType([
            T.StructField("audit_event_id", T.StringType(), False),
            T.StructField("entity_type", T.StringType(), False),
            T.StructField("entity_id", T.StringType(), False),
            T.StructField("action", T.StringType(), False),
            T.StructField("occurred_at", T.TimestampType(), False),
            T.StructField("actor_type", T.StringType(), False),
        ]),
    },
}

MUTABLE_TECHNICAL_COLUMNS = {
    "record_version": T.IntegerType(),
    "last_modified_at": T.TimestampType(),
    "last_confirmation_id": T.StringType(),
    "pending_reconciliation": T.BooleanType(),
    "is_fixture": T.BooleanType(),
}

MUTABLE_TABLES = {"reservations", "restaurant_reservations", "billing_disputes", "special_requests", "maintenance_requests", "work_orders"}

PROPERTY_CURRENCY_TABLES = {"reservations": "currency", "folios": "currency", "payments": "currency", "daily_rates": "currency", "property_daily_metrics": "currency"}

FIXTURE_KEYS = {
    "reservation": {"family_accessibility", "rate_change", "cancellable", "nonrefundable", "folio_dispute", "restaurant"},
    "request": {"prearrival_request", "maintenance"},
}


def table_path(workspace_id: str, lakehouse_id: str, table_name: str) -> str:
    return f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables/{table_name}"


def files_path(workspace_id: str, lakehouse_id: str, relative_path: str) -> str:
    trimmed = relative_path.lstrip("/")
    return f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Files/{trimmed}"


def resolve_local_or_abfss(source_root: str, child: str) -> str:
    if source_root.startswith("abfss://"):
        return source_root.rstrip("/") + "/" + child
    return str((Path(source_root) / child).resolve())


def read_source_table(source_root: str, table_name: str) -> DataFrame:
    spec = SOURCE_TABLE_SPECS[table_name]
    path = resolve_local_or_abfss(source_root, f"{table_name}.csv")
    return spark.read.option("header", True).schema(spec["schema"]).csv(path)


def read_json_document(source_root: str, child: str) -> dict[str, object]:
    path = resolve_local_or_abfss(source_root, child)
    if path.startswith("abfss://"):
        text = notebookutils.fs.head(path, 10_000_000)
    else:
        text = Path(path).read_text(encoding="utf-8")
    return json.loads(text)


def assert_no_duplicates(df: DataFrame, keys: list[str], label: str) -> None:
    duplicates = df.groupBy(*keys).count().where(F.col("count") > 1).limit(1).count()
    if duplicates:
        raise ValueError(f"Duplicate keys detected in {label}: {keys}")


def assert_row_count(df: DataFrame, expected: int, label: str) -> None:
    actual = df.count()
    if actual != expected:
        raise ValueError(f"{label} row count mismatch: expected {expected}, found {actual}")


def assert_foreign_key(child: DataFrame, child_key: str, parent: DataFrame, parent_key: str, label: str) -> None:
    missing = child.select(child_key).distinct().join(parent.select(parent_key).distinct(), F.col(child_key) == F.col(parent_key), "left_anti").limit(1).count()
    if missing:
        raise ValueError(f"Foreign-key validation failed for {label}: {child_key} -> {parent_key}")


def assert_date_window(df: DataFrame, column_name: str, label: str) -> None:
    bounds = df.select(F.min(column_name).alias("min_date"), F.max(column_name).alias("max_date")).first()
    min_date = bounds["min_date"]
    max_date = bounds["max_date"]
    if min_date is None or max_date is None or min_date < DATE_START or max_date > DATE_END:
        raise ValueError(f"{label}.{column_name} is outside the supported date window {DATE_START}..{DATE_END}")


def assert_property_currency(df: DataFrame, currency_column: str, properties_df: DataFrame, label: str) -> None:
    invalid = (
        df.alias("fact")
        .join(properties_df.select("property_id", F.col("currency").alias("property_currency")).alias("dim"), "property_id", "inner")
        .where(F.col(currency_column) != F.col("property_currency"))
        .limit(1)
        .count()
    )
    if invalid:
        raise ValueError(f"Currency mismatch detected in {label}")


def add_mutation_metadata(df: DataFrame, confirmation_id: str | None = None, pending_reconciliation: bool = False, is_fixture: bool = True) -> DataFrame:
    result = df
    if "record_version" not in result.columns:
        result = result.withColumn("record_version", F.lit(1))
    if "last_modified_at" not in result.columns:
        result = result.withColumn("last_modified_at", F.current_timestamp())
    if "last_confirmation_id" not in result.columns:
        result = result.withColumn("last_confirmation_id", F.lit(confirmation_id).cast(T.StringType()))
    if "pending_reconciliation" not in result.columns:
        result = result.withColumn("pending_reconciliation", F.lit(pending_reconciliation))
    if "is_fixture" not in result.columns:
        result = result.withColumn("is_fixture", F.lit(is_fixture))
    return result


def overwrite_delta(df: DataFrame, workspace_id: str, lakehouse_id: str, table_name: str) -> None:
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(table_path(workspace_id, lakehouse_id, table_name)))


def read_delta(workspace_id: str, lakehouse_id: str, table_name: str) -> DataFrame:
    return spark.read.format("delta").load(table_path(workspace_id, lakehouse_id, table_name))


def merge_single_row(workspace_id: str, lakehouse_id: str, table_name: str, key_column: str, payload_df: DataFrame, update_assignments: dict[str, str], insert_assignments: dict[str, str]) -> None:
    delta = DeltaTable.forPath(spark, table_path(workspace_id, lakehouse_id, table_name))
    (delta.alias("target").merge(payload_df.alias("source"), f"target.{key_column} = source.{key_column}")
        .whenMatchedUpdate(set=update_assignments)
        .whenNotMatchedInsert(values=insert_assignments)
        .execute())


def request_hash(payload_json: str) -> str:
    return hashlib.sha256(payload_json.encode("utf-8")).hexdigest()

In [ ]:
manifest = read_json_document(source_root, "manifest.json")
fixture_manifest = read_json_document(source_root, "../fixtures/demo_scenarios.json") if not source_root.startswith("abfss://") else json.loads(notebookutils.fs.head(source_root.rstrip("/") + "/../fixtures/demo_scenarios.json", 10_000_000))
required_tables = set(manifest["counts"].keys())
loaded: dict[str, DataFrame] = {}
for table_name in sorted(required_tables):
    loaded[table_name] = read_source_table(source_root, table_name)
    assert_no_duplicates(loaded[table_name], list(SOURCE_TABLE_SPECS[table_name]["primary_key"]), table_name)
    assert_row_count(loaded[table_name], int(manifest["counts"][table_name]), table_name)

properties_df = loaded["properties"]
room_types_df = loaded["room_types"]
rooms_df = loaded["rooms"]
reservations_df = loaded["reservations"]
stays_df = loaded["stays"]
folios_df = loaded["folios"]
transactions_df = loaded["folio_transactions"]
payments_df = loaded["payments"]
requests_df = loaded["special_requests"]
maintenance_df = loaded["maintenance_requests"]
work_orders_df = loaded["work_orders"]

assert_foreign_key(room_types_df, "property_id", properties_df, "property_id", "room_types")
assert_foreign_key(rooms_df, "property_id", properties_df, "property_id", "rooms")
assert_foreign_key(rooms_df, "room_type_id", room_types_df, "room_type_id", "rooms")
assert_foreign_key(reservations_df, "guest_id", loaded["guests"], "guest_id", "reservations")
assert_foreign_key(reservations_df, "property_id", properties_df, "property_id", "reservations")
assert_foreign_key(reservations_df, "room_type_id", room_types_df, "room_type_id", "reservations")
assert_foreign_key(stays_df, "reservation_id", reservations_df, "reservation_id", "stays")
assert_foreign_key(stays_df, "room_id", rooms_df, "room_id", "stays")
assert_foreign_key(loaded["restaurant_reservations"], "outlet_id", loaded["outlets"], "outlet_id", "restaurant_reservations")
assert_foreign_key(folios_df, "stay_id", stays_df, "stay_id", "folios")
assert_foreign_key(transactions_df, "folio_id", folios_df, "folio_id", "folio_transactions")
assert_foreign_key(payments_df, "folio_id", folios_df, "folio_id", "payments")
assert_foreign_key(requests_df, "reservation_id", reservations_df, "reservation_id", "special_requests")
assert_foreign_key(maintenance_df, "stay_id", stays_df, "stay_id", "maintenance_requests")
assert_foreign_key(work_orders_df, "maintenance_request_id", maintenance_df, "maintenance_request_id", "work_orders")
for table_name, spec in SOURCE_TABLE_SPECS.items():
    for column_name in spec.get("date_columns", []):
        assert_date_window(loaded[table_name], column_name, table_name)

for table_name, currency_column in PROPERTY_CURRENCY_TABLES.items():
    assert_property_currency(loaded[table_name], currency_column, properties_df, table_name)

room_property_mismatch = (
    rooms_df.alias("room")
    .join(room_types_df.alias("type"), "room_type_id")
    .where(F.col("room.property_id") != F.col("type.property_id"))
    .limit(1)
    .count()
)
if room_property_mismatch:
    raise ValueError("Room/property consistency validation failed.")

folio_reconciliation = (
    folios_df.alias("folio")
    .join(transactions_df.groupBy("folio_id").agg(F.sum("amount").alias("transaction_total")), "folio_id", "left")
    .join(payments_df.groupBy("folio_id").agg(F.sum("amount").alias("payment_total")), "folio_id", "left")
    .withColumn("transaction_total", F.coalesce(F.col("transaction_total"), F.lit(Decimal("0.00"))))
    .withColumn("payment_total", F.coalesce(F.col("payment_total"), F.lit(Decimal("0.00"))))
    .where((F.col("transaction_total") != (F.col("total_charges") + F.col("total_adjustments"))) | (F.col("payment_total") != F.col("total_payments")) | ((F.col("total_charges") + F.col("total_adjustments") - F.col("total_payments") - F.col("total_refunds")) != F.col("balance")))
    .limit(1)
    .count()
)
if folio_reconciliation:
    raise ValueError("Folio reconciliation validation failed.")

rate_plans_df = loaded["daily_rates"].select("property_id", "rate_plan", "currency").distinct()
reservation_guests_df = add_mutation_metadata(
    reservations_df.select(
        F.concat_ws("::", F.col("reservation_id"), F.col("guest_id")).alias("reservation_guest_id"),
        "reservation_id",
        "guest_id",
        F.lit("primary").alias("relationship_type"),
    ),
    is_fixture=True,
)
room_assignments_df = add_mutation_metadata(
    stays_df.select(
        F.col("stay_id").alias("room_assignment_id"),
        "stay_id",
        "room_id",
        "property_id",
        F.col("actual_check_in").alias("assigned_at"),
        F.col("actual_check_out").alias("released_at"),
    ),
    is_fixture=True,
)
adjustments_df = add_mutation_metadata(
    transactions_df.where(F.col("transaction_type") == "adjustment").select(
        F.col("folio_transaction_id").alias("adjustment_id"),
        "folio_id",
        "amount",
        "currency",
        F.col("posted_at").alias("adjusted_at"),
        F.col("description").alias("reason"),
    ),
    is_fixture=True,
)
action_confirmations_df = loaded["action_confirmations"].withColumn("idempotency_key", F.lit(None).cast(T.StringType())).withColumn("expected_record_version", F.lit(None).cast(T.IntegerType())).withColumn("correlation_id", F.lit(correlation_id)).withColumn("notebook_name", F.lit("HotelInitialLoad")).withColumn("result_json", F.lit('{"seed":true}')).withColumn("pending_reconciliation", F.lit(False))
idempotency_records_df = loaded["idempotency_records"].withColumn("confirmation_id", F.lit(None).cast(T.StringType())).withColumn("request_hash", F.lit(None).cast(T.StringType())).withColumn("status", F.lit("seeded")).withColumn("notebook_name", F.lit("HotelInitialLoad"))
audit_events_df = loaded["audit_events"].withColumn("confirmation_id", F.lit(None).cast(T.StringType())).withColumn("idempotency_key", F.lit(None).cast(T.StringType())).withColumn("payload_json", F.lit('{"seed":true}'))
action_sagas_df = spark.createDataFrame([], T.StructType([
    T.StructField("saga_id", T.StringType(), False),
    T.StructField("action_type", T.StringType(), False),
    T.StructField("confirmation_id", T.StringType(), False),
    T.StructField("guest_services_entity_id", T.StringType(), True),
    T.StructField("operations_entity_id", T.StringType(), True),
    T.StructField("state", T.StringType(), False),
    T.StructField("pending_reconciliation", T.BooleanType(), False),
    T.StructField("created_at", T.TimestampType(), False),
    T.StructField("updated_at", T.TimestampType(), False),
    T.StructField("details_json", T.StringType(), True),
]))

operations_tables = {
    "properties": properties_df,
    "room_types": room_types_df,
    "rooms": rooms_df,
    "amenities": loaded["amenities"],
    "property_amenities": loaded["property_amenities"],
    "daily_inventory": loaded["daily_inventory"],
    "daily_rates": loaded["daily_rates"],
    "property_daily_metrics": loaded["property_daily_metrics"],
    "outlets": loaded["outlets"],
    "rate_plans": rate_plans_df,
    "maintenance_requests": add_mutation_metadata(maintenance_df),
    "work_orders": add_mutation_metadata(work_orders_df),
}

guest_service_tables = {
    "guests": loaded["guests"],
    "reservations": add_mutation_metadata(reservations_df),
    "reservation_guests": reservation_guests_df,
    "stays": stays_df.withColumn("is_fixture", F.lit(True)),
    "room_assignments": room_assignments_df,
    "restaurant_reservations": add_mutation_metadata(loaded["restaurant_reservations"]),
    "folios": folios_df.withColumn("is_fixture", F.lit(True)),
    "folio_transactions": transactions_df.withColumn("is_fixture", F.lit(True)),
    "payments": payments_df.withColumn("is_fixture", F.lit(True)),
    "adjustments": adjustments_df,
    "billing_disputes": add_mutation_metadata(loaded["billing_disputes"]),
    "special_requests": add_mutation_metadata(requests_df),
    "action_confirmations": action_confirmations_df,
    "idempotency_records": idempotency_records_df,
    "audit_events": audit_events_df,
    "action_sagas": action_sagas_df,
}

if reset_before_load:
    for table_name in operations_tables:
        notebookutils.fs.rm(table_path(workspace_id, hotel_operations_lakehouse_id, table_name), True)
    for table_name in guest_service_tables:
        notebookutils.fs.rm(table_path(workspace_id, hotel_guest_services_lakehouse_id, table_name), True)

for table_name, dataframe in operations_tables.items():
    overwrite_delta(dataframe, workspace_id, hotel_operations_lakehouse_id, table_name)
for table_name, dataframe in guest_service_tables.items():
    overwrite_delta(dataframe, workspace_id, hotel_guest_services_lakehouse_id, table_name)

fixture_scenarios = fixture_manifest["scenarios"]
reservation_fixture_ids = {item["reservation_id"] for item in fixture_scenarios if "reservation_id" in item}
request_fixture_ids = {item["request_id"] for item in fixture_scenarios if "request_id" in item}
if reservations_df.where(F.col("reservation_id").isin(list(reservation_fixture_ids))).select("reservation_id").distinct().count() != len(reservation_fixture_ids - {"RR-DEMO-DINING"}):
    raise ValueError("Demo reservation fixtures are missing from reservations.")
if requests_df.where(F.col("special_request_id").isin(list(request_fixture_ids & {"SR-DEMO-PREARRIVAL"}))).select("special_request_id").distinct().count() != 1:
    raise ValueError("Pre-arrival special-request fixture is missing.")
if maintenance_df.where(F.col("maintenance_request_id") == "MR-DEMO-HVAC").limit(1).count() != 1:
    raise ValueError("Maintenance fixture MR-DEMO-HVAC is missing.")

result = {
    "correlation_id": correlation_id,
    "workspace_id": workspace_id,
    "hotel_operations_tables": {name: df.count() for name, df in operations_tables.items()},
    "hotel_guest_services_tables": {name: df.count() for name, df in guest_service_tables.items()},
    "date_window": {"start": DATE_START.isoformat(), "end": DATE_END.isoformat()},
    "load_mode": load_mode,
}

In [ ]:
notebookutils.notebook.exit(json.dumps(result, default=str))